# wf_v5.0 GPU session 2: G5 rerun (Step 4 variant ARM, guided)
The first G5 run got finish_reason "error" on every guided Step 4 call and used the unguided fallback, so it is not comparable with v4.9.
This session runs only G5 again, into `results/wf_v50/g5/<set>/step4_ARMg/`, on the Step 4 inputs from session 1 (in git). Branch `wf-v5.0`.

- Cells 0-6: setup as in session 1 (cell 2 installs, then restart the session, rerun cells 0-1 and skip cell 2; cell 3 before any `s1a` import).
- Cell 7: data check. **Cell 8: guided smoke test**: if it fails, stop, copy its output and tell Claude (do not run cell 9).
- Cell 9: G5 (6 chains); it stops if any call ran unguided. Cell 10: settings, then **Runtime > Disconnect and delete runtime**, tell Claude "GPU done".
- Run on a **fresh runtime** (not the session-1 runtime).


In [ ]:
#@title 0. GPU check: nvidia-smi, assert an A100 with 40 GB or more
import re, subprocess
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr.strip())
assert smi.returncode == 0 and smi.stdout.strip(), "no GPU: set the runtime type to A100"
name, mib = [x.strip() for x in smi.stdout.splitlines()[0].split(",")]
assert re.search(r"A100|H100|H200|B200", name) and int(mib) >= 40000, f"need an A100 with 40 GB or more, got {name} {mib} MiB"
print("GPU ok:", name, mib, "MiB")


In [ ]:
#@title 1. Tokens, clone/pull wf-v5.0, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v5.0"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Data and the loop-2 models (MultiWOZ zip + ids from HF, GLiNER, deberta-v3-base, bge-m3; pinned revisions)
import json, os
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download
MODELS = Path("/content/models")
os.environ["CHK_MODELS"] = str(MODELS)          # read by chunking/chk/config.py and s1a/spans.py at import
DATA = Path("/content/attribute-extraction/chunking/data")
for f in ("MULTIWOZ2.4.zip", "conversation_ids.txt"):
    if not (DATA / f).exists():
        hf_hub_download("database-lab/attribute-extraction-data", f"chunking/{f}", repo_type="dataset", local_dir=DATA.parent)
snapshot_download("urchade/gliner_medium-v2.1", revision="40ec419335d09393f298636f471328b722c6da9e", local_dir=MODELS / "gliner_medium-v2.1")
snapshot_download("microsoft/deberta-v3-base", revision="8ccc9b6f36199bec6961081d44eb72fb3f7353f3", local_dir=MODELS / "deberta-v3-base",
                  allow_patterns=["config.json", "spm.model", "tokenizer_config.json"])
snapshot_download("BAAI/bge-m3", revision="5617a9f61b028005a4858fdac845db406aefb181", local_dir=MODELS / "bge-m3")
cfg_path = MODELS / "gliner_medium-v2.1" / "gliner_config.json"   # point GLiNER at the local backbone, as on the laptop
cfg = json.loads(cfg_path.read_text())
cfg["model_name"] = str(MODELS / "deberta-v3-base")
cfg_path.write_text(json.dumps(cfg, indent=2))
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a"); sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
from s1a import spans as SP
g = SP.load()
print("GLiNER ok:", g.inference(["I need a hotel with free wifi."], ["internet access"], threshold=0.3))
del g

In [ ]:
#@title 4. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 5. Load the model (gpu_memory_utilization leaves room for bge-m3)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.6  #@param {type:"number"}
BATCH_INVARIANT = True  #@param {type:"boolean"}
import os
os.environ["VLLM_BATCH_INVARIANT"] = "1" if BATCH_INVARIANT else "0"   # read when vllm is imported
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
import time
_t0 = _T_SESSION = time.time()   # the GPU session clock starts with the model load
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)
from s1a import log_v44 as L44
print("model load:", L44.save_load_time("results/wf_v50/model_load_g5b.json", time.time() - _t0, MODEL, REVISION))


In [ ]:
#@title 6. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/log_v44.VllmBackend44)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
from s1a import log_v44 as L44
backend = L44.VllmBackend44(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 7. Data check: the G5 inputs from session 1 (git)
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import gpu_v50 as G
need = [G.G48.input_dir(d, G.G5) / r / "a.columns.json" for d, r in G.SETS.items()]
need += [G.Y3 / d / f for d in G.SETS for f in ("entities.json", "a_prompts")]
missing = [str(p) for p in need if not p.exists()]
print("missing:", missing or "none")
assert not missing, "files missing: stop and tell Claude"
print(G.G48.config("ARMg"))


In [ ]:
#@title 8. Guided smoke test (must print finish stop and {"a": "x"} or {"a": "y"})
fin, txt = G.guided_smoke(llm, sampling, backend.chat)
print("finish:", fin, "| text:", txt)
assert fin == "stop" and txt.strip().startswith("{"), "guided decoding is broken in this session: stop and tell Claude (copy this output)"


In [ ]:
#@title 9. G5 rerun: Step 4 variant ARMg, seeds 0-2, dev50 and dev50b -> results/wf_v50/g5/<set>/step4_ARMg
MAX_CALLS = 140  #@param {type:"integer"}
import subprocess
from s1a import log_v44 as L44
res = G.g5(backend)
for (var, dev), rows in res.items():
    folder = G.G5 / dev / f"step4_{var}"
    calls = len(L44.call_rows(folder))
    kinds = {k: sum(x["repair_kinds"].get(k, 0) for x in rows) for k in ("p2_reask", "p2_reask2", "p2_missing_column", "p2_unknown_column")}
    bad = G.unguided_rows(folder)
    print(var, dev, [(x["seed"], x["ok"], x["tables"], x["error"]) for x in rows], "| calls", calls, "| unguided", len(bad), "|", kinds)
    assert calls <= MAX_CALLS, f"{var} {dev}: more calls than MAX_CALLS: stop and tell Claude"
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v50 G5 rerun ARMg", "wf_v50"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
assert not any(G.unguided_rows(G.G5 / d / "step4_ARMg") for d in G.SETS), "some calls ran unguided: tell Claude"


In [ ]:
#@title 10. vLLM settings, GPU session time -> results/wf_v50/settings_g5b.json, then disconnect
import json, os, time, subprocess
def _seed():
    try:
        return llm.llm_engine.model_config.seed
    except Exception as e:
        return f"not readable: {type(e).__name__}"
info = {"chains": {f"{v} {d}": [(x["run"], x["ok"], x["tables"]) for x in rows] for (v, d), rows in res.items()},
        "vllm": vllm.__version__, "batch": BATCH, "batch_invariant_env": os.environ.get("VLLM_BATCH_INVARIANT"), "engine_seed": _seed(),
        "temperature": 0, "model": MODEL, "revision": REVISION, "smoke": [fin, txt],
        "unguided": {d: len(G.unguided_rows(G.G5 / d / "step4_ARMg")) for d in G.SETS},
        "gpu_session_seconds": round(time.time() - _T_SESSION, 1), "model_load_seconds": L44.load_time("results/wf_v50/model_load_g5b.json")}
open("results/wf_v50/settings_g5b.json", "w").write(json.dumps(info, indent=1, default=str) + "
")
print(json.dumps(info, indent=1, default=str))
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v50 G5 rerun settings", "wf_v50"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("Done: Runtime > Disconnect and delete runtime, then tell Claude 'GPU done'")
